# LIBRARY

In [49]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
)
from sktime.transformations.series.adapt import TabularToSeriesAdaptor
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV

from sktime.classification.shapelet_based import ShapeletTransformClassifier
from sktime.transformations.panel.shapelet_transform import RandomShapeletTransform
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sktime.classification.distance_based import KNeighborsTimeSeriesClassifier
from scikitplot.metrics import plot_roc

from sklearn.model_selection import GroupShuffleSplit
import numpy as np
import pandas as pd
import gzip 
import pickle 
from collections import Counter
import matplotlib.pyplot as plt
import gzip
import pickle
import sys
import numpy as np
import warnings
import pandas as pd
from scipy.stats import levene
from statsmodels.tsa.stattools import adfuller, kpss
from sklearn.preprocessing import RobustScaler
from statsmodels.tsa.seasonal import STL
from statsmodels.tools.sm_exceptions import InterpolationWarning
warnings.simplefilter("ignore", InterpolationWarning)
warnings.simplefilter("ignore", UserWarning)


# DATAFRAMES

In [43]:
with gzip.open("../1.DATASET/CMI_timeseries_personalized.pkl.gz", "rb") as f:
	 CMI_timeseries_personalized = pickle.load(f)

In [44]:
# Check how many subjects/time series
print(f"Number of time series: {len(CMI_timeseries_personalized)}")

Number of time series: 4307


## Cleaning

In [45]:
df = CMI_timeseries_personalized[0].copy()
df.columns

Index(['X', 'Y', 'Z', 'enmo', 'anglez', 'non-wear_flag', 'light',
       'battery_voltage', 'weekday', 'quarter', 'relative_date_PCIAT', 'id',
       'sii_binary'],
      dtype='object')

In [46]:
cols_to_drop = ["battery_voltage", "timestamp", "quarter", "relative_date_PCIAT"]
data_clean   = [df.drop(columns=cols_to_drop, errors="ignore") for df in CMI_timeseries_personalized]

print(f"Columns remaining: {data_clean[0].columns.tolist()}")
print(f"Total subjects   : {len(data_clean)}")

Columns remaining: ['X', 'Y', 'Z', 'enmo', 'anglez', 'non-wear_flag', 'light', 'weekday', 'id', 'sii_binary']
Total subjects   : 4307


## id check

In [48]:
all_labels = [df["sii_binary"].iloc[0] for df in data_clean]
all_ids    = [df["id"].iloc[0]         for df in data_clean]

# Filter missing labels
valid_idx    = [i for i, label in enumerate(all_labels) if not pd.isna(label)]
valid_labels = np.array([all_labels[i] for i in valid_idx])
valid_ids    = np.array([all_ids[i]    for i in valid_idx])

print(f"Total datasets          : {len(data_clean)}")
print(f"Valid (non-null label)  : {len(valid_idx)}")
print(f"Unique subject IDs      : {len(set(valid_ids))}")

Total datasets          : 4307
Valid (non-null label)  : 4307
Unique subject IDs      : 372


## split 

In [50]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_pos, test_pos = next(gss.split(
    X=valid_idx,
    y=valid_labels,
    groups=valid_ids
))

train_idx = [valid_idx[i] for i in train_pos]
test_idx  = [valid_idx[i] for i in test_pos]

train_dfs = [data_clean[i] for i in train_idx]
test_dfs  = [data_clean[i] for i in test_idx]

In [51]:
# ── Sanity checks ─────────────────────────────────────────────────────────────
train_ids = set(df["id"].iloc[0] for df in train_dfs)
test_ids  = set(df["id"].iloc[0] for df in test_dfs)
overlap   = train_ids & test_ids

print(f"\nTrain datasets  : {len(train_dfs)}")
print(f"Test datasets   : {len(test_dfs)}")
print(f"Total           : {len(train_dfs) + len(test_dfs)}")
print(f"Subject ID overlap (must be 0): {len(overlap)}")


Train datasets  : 3298
Test datasets   : 1009
Total           : 4307
Subject ID overlap (must be 0): 0


# NORMALIZATION

In [52]:
signals = ["X", "Y", "Z", "enmo", "anglez", "light", "non-wear_flag", "weekday"]

In [53]:
def fit_global_scalers(df_list, signals):
    """Fit one RobustScaler per signal using TRAIN data only."""
    global_scalers = {}
    print("Fitting global scalers on train data...")
    for signal in signals:
        all_values = []
        for df in df_list:
            if signal not in df.columns:
                continue
            y = df[signal].values
            if np.std(y) < 0.0001 or np.ptp(y) == 0:
                continue
            all_values.append(y)
        if not all_values:
            continue
        population = np.concatenate(all_values).reshape(-1, 1)
        q25, q75   = np.percentile(population, [25, 75])
        iqr        = q75 - q25
        signal_std = np.std(population)
        if iqr > 1e-4:
            scaler = RobustScaler()
            scaler.fit(population)
            global_scalers[signal] = ("robust", scaler)
            print(f"  {signal:<18} → RobustScaler   (IQR={iqr:.4f})")
        elif signal_std > 1e-6:
            global_scalers[signal] = ("std_fallback", np.median(population), signal_std)
            print(f"  {signal:<18} → Std fallback   (IQR too small)")
        else:
            global_scalers[signal] = ("zero", None)
            print(f"  {signal:<18} → Zeroed")
    return global_scalers


In [54]:
def apply_global_scalers(df_list, global_scalers):
    """Apply pre-fitted scalers to a list of DataFrames."""
    scaled_list = []
    for df in df_list:
        target_df = df.copy()
        for signal, scaler_info in global_scalers.items():
            if signal not in target_df.columns:
                continue
            y    = target_df[signal].values.copy()
            kind = scaler_info[0]
            if kind == "robust":
                _, scaler = scaler_info
                y_scaled  = scaler.transform(y.reshape(-1, 1)).flatten()
            elif kind == "std_fallback":
                _, median, std = scaler_info
                y_scaled  = (y - median) / std
            else:
                y_scaled  = np.zeros_like(y)
            target_df[signal] = y_scaled
        scaled_list.append(target_df)
    return scaled_list


In [55]:

global_scalers = fit_global_scalers(train_dfs, signals)
train_scaled   = apply_global_scalers(train_dfs, global_scalers)
test_scaled    = apply_global_scalers(test_dfs,  global_scalers)

print(f"\nTrain scaled: {len(train_scaled)} | Test scaled: {len(test_scaled)}")

Fitting global scalers on train data...
  X                  → RobustScaler   (IQR=0.8217)
  Y                  → RobustScaler   (IQR=0.4166)
  Z                  → RobustScaler   (IQR=0.4350)
  enmo               → RobustScaler   (IQR=0.0463)
  anglez             → RobustScaler   (IQR=5.6673)
  light              → RobustScaler   (IQR=1.8472)
  non-wear_flag      → RobustScaler   (IQR=1.0000)

Train scaled: 3298 | Test scaled: 1009
